<div style="background: linear-gradient(135deg, #0f172a 0%, #1e40af 100%); color: white; padding: 32px 40px; border-radius: 14px;"><div style="font-size: 0.85em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.8;">Projet DataViz &mdash; Étape 4</div><div style="font-size: 1.9em; font-weight: 800; margin-top: 6px;">Score de tension d’accès</div><div style="font-size: 1.05em; margin-top: 10px; opacity: 0.92;">Un indice unique par commune (0&ndash;100) pour aider les élus à prioriser. Version à 2 KPIs ; l’âge des médecins viendra l’enrichir.</div></div>

## Principe

On combine les indicateurs d’offre médicale en **un score de tension** : **plus il est haut, plus l’accès est difficile** (donc plus la commune est prioritaire).

**Choix de méthode, assumés et explicables :**

1. **Sens** — un APL ou une densité élevés = *bon* accès. On **inverse** pour que le score mesure la *tension*.
2. **Normalisation min-max winsorisée** (bornée aux percentiles 2 % / 98 %) : chaque indicateur est ramené sur 0–1, sans qu’un cas extrême (ex. Paris) n’écrase toute l’échelle.
3. **Pondération** — 2 KPIs à parts égales : **½ généralistes + ½ spécialistes** (moyenne des 4 spécialités). Quand le KPI âge sera ajouté, on passera à ⅓ / ⅓ / ⅓.
4. **Deux lectures** : un score **absolu** (comparaison régionale) et un rang **relatif intra-EPCI** (« parmi mes communes voisines, lesquelles sont en tension ? ») — pour éviter un palmarès national qui concentrerait tous les efforts au même endroit.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.facecolor"] = "white"; plt.rcParams["savefig.facecolor"] = "white"

df = pd.read_parquet("data/processed/communes_idf_consolide.parquet")
SPE = ["densite_cardio_100k", "densite_dermato_100k", "densite_ophtalmo_100k", "densite_gyneco_100k"]
print(df.shape)

## 1. Normalisation → composantes de tension

Chaque indicateur devient une *tension* entre 0 (le mieux doté de la région) et 1 (le plus tendu).

In [ ]:
def tension(s, lo=0.02, hi=0.98):
    """Min-max winsorisé puis inversé : 0 = bien doté, 1 = très tendu."""
    a, b = s.quantile(lo), s.quantile(hi)
    x = (s.clip(a, b) - a) / (b - a)
    return 1 - x

df["t_generaliste"] = tension(df["apl_generaliste"])
for c in SPE:
    df["t_" + c.split("_")[1]] = tension(df[c])
spe_t = ["t_cardio", "t_dermato", "t_ophtalmo", "t_gyneco"]
df["tension_specialistes"] = df[spe_t].mean(axis=1)

df[["nom_commune", "t_generaliste"] + spe_t + ["tension_specialistes"]].head().round(2)

## 2. Score composite (0–100)

In [ ]:
df["score_tension"] = 100 * (0.5 * df["t_generaliste"] + 0.5 * df["tension_specialistes"])
df["score_tension"] = df["score_tension"].round(1)

print(df["score_tension"].describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 4.3))
sns.histplot(df["score_tension"], bins=40, color="#dc2626", ax=ax)
ax.axvline(df["score_tension"].median(), color="#0f172a", ls="--",
           label=f"médiane {df.score_tension.median():.0f}")
ax.set(xlabel="Score de tension (0 = bien doté, 100 = très tendu)", ylabel="Communes",
       title="Distribution du score de tension d’accès — Île-de-France")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Lecture absolue — les communes les plus tendues de la région

In [ ]:
cols = ["nom_commune", "dep", "epci_nom", "pop_commune", "apl_generaliste",
        "tension_specialistes", "score_tension"]
df.nlargest(15, "score_tension")[cols].reset_index(drop=True).round(2)

## 4. Lecture relative intra-EPCI

Pour chaque commune, son **rang en percentile au sein de son EPCI** (1 = la plus tendue de son intercommunalité). C’est la vue qui aide un maire / président d’EPCI à arbitrer **localement**.

In [ ]:
df["rang_epci_pct"] = (df.groupby("epci_siren")["score_tension"]
                         .rank(pct=True, method="average")).round(2)

# exemple : une grande intercommunalité, communes classées par tension relative
ex = df[df.epci_nom.str.contains("Grand Paris Sud", na=False)]
if len(ex) == 0:
    ex = df[df.epci_siren == df.epci_siren.value_counts().idxmax()]
print("EPCI exemple :", ex.epci_nom.iloc[0], f"({len(ex)} communes)")
ex.sort_values("score_tension", ascending=False)[
    ["nom_commune", "apl_generaliste", "score_tension", "rang_epci_pct"]].head(10).round(2)

## 5. Synthèse par territoire

In [ ]:
par_dep = df.groupby("dep").agg(
    communes=("code_insee", "size"),
    population=("pop_commune", "sum"),
    score_moyen=("score_tension", "mean"),
    apl_gen_moyen=("apl_generaliste", "mean")).round(1)
par_dep.sort_values("score_moyen", ascending=False)

## 6. Export pour le dashboard

Table enrichie (communes + composantes + score) prête pour Tableau Public.

In [ ]:
export_cols = [
    "code_insee", "nom_commune", "dep", "epci_siren", "epci_nom", "epci_nature",
    "pop_commune", "apl_generaliste",
    "densite_cardio_100k", "densite_dermato_100k", "densite_ophtalmo_100k", "densite_gyneco_100k",
    "t_generaliste", "tension_specialistes", "score_tension", "rang_epci_pct"]
out = df[export_cols].copy()
out.to_csv("data/processed/communes_idf_score.csv", index=False, encoding="utf-8-sig")
out.to_parquet("data/processed/communes_idf_score.parquet", index=False)
print("[OK] data/processed/communes_idf_score.csv  (" + str(len(out)) + " communes, "
      + str(len(export_cols)) + " colonnes)")

## Limites & suite

- Score à **2 KPIs** pour l’instant ; l’ajout de l’**âge des médecins** (KPI 3, data.drees) fera passer la pondération à ⅓ / ⅓ / ⅓ et affinera le risque à 5–10 ans.
- La densité spécialistes est **départementale** : à l’intérieur d’un département elle est constante, donc la variation intra-EPCI vient surtout des généralistes — cohérent avec la vue maire.
- Pondération **1/2–1/2** retenue par défaut ; à challenger avec le formateur selon le pouvoir discriminant des indicateurs (la dermato ressort comme la plus discriminante en EDA).

**Étape suivante** → construction du **dashboard Tableau Public** (2 niveaux de zoom) à partir de `communes_idf_score.csv`.